# JEPA

> A LEWM-based architecture for learning representations of subseqt observations from current ones. 

In [1]:
#| default_exp models.jepa

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import torch
from torch import nn
from torch.nn import functional as F
from einops import rearrange


In [ ]:
#| export
def modulate(x, shift, scale):
    """AdaLN-zero modulation"""
    return x * (1 + scale) + shift

## Messages

A receiver hears from up to `N` other agents. What it knows of each, at each step, is that sender's latest message
it received: the message's 49 VQ tokens (the sender's 7 x 7 view), the sender's pose when it sent it relative to
where the receiver is now (`relative_pose`: cells ahead, cells to the right, heading; an old message is placed from
the receiver's current pose, so the receiver need not track how it moved since), and its age in steps. Nothing about
the channel itself: the channel shows only in which messages arrived. `latest_messages` builds these per-step
snapshots from the messages sent at each step, which of them arrived, and the agents' poses.

In [ ]:
#| export
DIR_VECTORS = ((1, 0), (0, 1), (-1, 0), (0, -1))                    # the grid's directions: right, down, left, up

def relative_pose(sender_pos, sender_dir, receiver_pos, receiver_dir):
    """Where a sender is and which way it faces, as the receiver sees it: `(..., 3)` integers (cells ahead, cells to
    the right, clockwise quarter turns from the receiver's heading). Positions `(..., 2)` as (x, y) cells (y down),
    directions as `DIR_VECTORS`' indices."""
    vec = torch.tensor(DIR_VECTORS, device=sender_pos.device)
    ahead = vec[receiver_dir]
    right = torch.stack([-ahead[..., 1], ahead[..., 0]], -1)
    d = sender_pos - receiver_pos
    return torch.stack([(d * ahead).sum(-1), (d * right).sum(-1), (sender_dir - receiver_dir) % 4], -1)

def latest_messages(tokens, received, sender, receiver, steps: int, max_age: int) -> dict:
    """The predictor's messages for the last `steps` of S steps: each sender's latest message received at most
    `max_age` steps before (age 0: sent this step), with the sender's pose when it sent it relative to the receiver's
    pose at the step it is read (`relative_pose`).
    tokens:   (B, S, N, L) the messages sent at each step (L tokens each)
    received: (B, S, N)    which of them arrived
    sender:   (B, S, N, 3) each sender's pose at each step: x, y (cells, y down), direction (`DIR_VECTORS`' index)
    receiver: (B, S, 3)    the receiver's pose at each step
    returns:  tokens (B, steps, N, L), received (B, steps, N), age (B, steps, N), pose (B, steps, N, 3)"""
    B, S, N = received.shape
    t = torch.arange(S, device=received.device).view(1, S, 1).expand(B, S, N)
    last = torch.where(received, t, torch.full_like(t, -1)).cummax(dim=1).values[:, -steps:]   # latest arrival
    age = t[:, -steps:] - last
    ok = (last >= 0) & (age <= max_age)
    index = last.clamp(min=0)

    def at_last(x):
        i = index.view(B, steps, N, *[1] * (x.dim() - 3)).expand(B, steps, N, *x.shape[3:])
        return torch.gather(x, 1, i)

    sent_from, now = at_last(sender), receiver[:, -steps:, None]
    pose = relative_pose(sent_from[..., :2], sent_from[..., 2], now[..., :2], now[..., 2])
    return {"tokens": at_last(tokens), "received": ok, "age": torch.where(ok, age, torch.zeros_like(age)), "pose": pose}

In [ ]:
#| hide
# a receiver at (5, 5) facing up: a sender 2 cells above is 2 ahead; one 2 cells to the right (x + 2) is 2 to its right
up, right = torch.tensor(3), torch.tensor(0)
assert relative_pose(torch.tensor([5, 3]), up, torch.tensor([5, 5]), up).tolist() == [2, 0, 0]
assert relative_pose(torch.tensor([7, 5]), right, torch.tensor([5, 5]), up).tolist() == [0, 2, 1]
assert relative_pose(torch.tensor([5, 7]), up, torch.tensor([5, 5]), right).tolist() == [0, 2, 3]

# one sender, 6 steps, a message sent at every step (the message of step s: all s), arrived at steps 1 and 4 only;
# the sender s cells above (5, 5) at step s, facing up; the receiver at (5, 5) facing up, at step 5 one cell up
tokens = torch.arange(6).view(1, 6, 1, 1).expand(1, 6, 1, 49)
received = torch.tensor([False, True, False, False, True, False]).view(1, 6, 1)
sender = torch.stack([torch.full((6,), 5), 5 - torch.arange(6), torch.full((6,), 3)], -1).view(1, 6, 1, 3)
receiver = torch.tensor([[5, 5, 3]] * 5 + [[5, 4, 3]]).view(1, 6, 3)
m = latest_messages(tokens, received, sender, receiver, steps=5, max_age=2)        # steps 1-5
assert m["received"][0, :, 0].tolist() == [True] * 5
assert m["age"][0, :, 0].tolist() == [0, 1, 2, 0, 1]
assert m["tokens"][0, :, 0, 0].tolist() == [1, 1, 1, 4, 4]
assert m["pose"][0, :, 0, 0].tolist() == [1, 1, 1, 4, 3]            # step 5: step 4's message, from where the receiver is now
m = latest_messages(tokens, received, sender, receiver, steps=5, max_age=1)        # step 3's message (age 2) has expired
assert m["received"][0, :, 0].tolist() == [True, True, False, True, True] and m["age"][0, 2, 0] == 0
m = latest_messages(tokens, received, sender, receiver, steps=6, max_age=2)        # step 0: nothing yet
assert not m["received"][0, 0, 0]

In [5]:
#| export
class SIGReg(torch.nn.Module):
    """Sketch Isotropic Gaussian Regularizer (single-GPU!)"""

    def __init__(self, knots=17, num_proj=1024):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt
        window = torch.exp(-t.square() / 2.0)
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj):
        """
        proj: (T, B, D)
        """
        # sample random projections
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        # compute the epps-pulley statistic
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        statistic = (err @ self.weights) * proj.size(-2)
        return statistic.mean() # average over projections and time
    

In [6]:
#| export
class DiscreteActionEncoder(nn.Module):
    def __init__(self, num_actions, action_emb_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_actions, action_emb_dim)
    
    def forward(self, action):
        """
        action: (B, T) integer indices
        returns: (B, T, action_emb_dim)
        """
        return self.embedding(action)
    

In [7]:
#| export
class FeedForward(nn.Module):
    """FeedForward network used in Transformers"""

    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [8]:
#| export
class Attention(nn.Module):
    """Scaled dot-product attention with causal masking"""

    def __init__(self, dim, heads=8, dim_head=64, dropout=0.0):
        super().__init__()
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == dim)
        self.heads = heads
        self.scale = dim_head**-0.5
        self.dropout = dropout
        self.norm = nn.LayerNorm(dim)
        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = (
            nn.Sequential(nn.Linear(inner_dim, dim), nn.Dropout(dropout))
            if project_out
            else nn.Identity()
        )

    def forward(self, x, causal=True):
        """
        x : (B, T, D)
        """
        x = self.norm(x)
        drop = self.dropout if self.training else 0.0
        qkv = self.to_qkv(x).chunk(3, dim=-1)  # q, k, v: (B, heads, T, dim_head)
        q, k, v = (rearrange(t, "b t (h d) -> b h t d", h=self.heads) for t in qkv)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=drop, is_causal=causal)
        out = rearrange(out, "b h t d -> b t (h d)")
        return self.to_out(out)



In [ ]:
#| export
class MessageConditionedBlock(nn.Module):
    """A transformer block with AdaLN-zero action conditioning that also reads the received messages: each step's
    state attends to that step's message tokens (and to a 'nothing received' token, always there). The cross
    attention's output projection starts at zero, so the block starts as the blind one."""

    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.0):
        super().__init__()
        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(dim, 6 * dim, bias=True)
        )
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout
        )
        nn.init.constant_(self.cross_attn.out_proj.weight, 0)
        nn.init.constant_(self.cross_attn.out_proj.bias, 0)
        self.cross_norm = nn.LayerNorm(dim)
        self.key_norm = nn.LayerNorm(dim)

    def forward(self, x, c, msg_keys=None, msg_mask=None, return_attention=False):
        """
        x, c:     (B, T, D)
        msg_keys: (B*T, K, D) each step's message tokens; msg_mask: (B*T, K), True where nothing was received
        """
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
            self.adaLN_modulation(c).chunk(6, dim=-1)
        )
        x = x + gate_msa * self.attn(modulate(self.norm1(x), shift_msa, scale_msa))

        attn_weights = None
        if msg_keys is not None:
            B, T, D = x.shape
            keys = self.key_norm(msg_keys)
            read, attn_weights = self.cross_attn(
                query=self.cross_norm(x).reshape(B * T, 1, D), key=keys, value=keys,
                key_padding_mask=msg_mask, need_weights=return_attention,
            )
            x = x + read.view(B, T, D)
            if attn_weights is not None:
                attn_weights = attn_weights.view(B, T, -1)

        x = x + gate_mlp * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))

        if return_attention:
            return x, attn_weights
        return x

In [11]:
#| export
class ConditionalBlock(nn.Module):
    """Transformer block with AdaLN-zero conditioning"""

    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.0):
        super().__init__()

        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(dim, 6 * dim, bias=True)
        )

        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
            self.adaLN_modulation(c).chunk(6, dim=-1)
        )
        x = x + gate_msa * self.attn(modulate(self.norm1(x), shift_msa, scale_msa))
        x = x + gate_mlp * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x



In [12]:
#| export
class Block(nn.Module):
    """Standard Transformer block"""

    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.0):
        super().__init__()

        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x



In [ ]:
#| export
class Transformer(nn.Module):
    """Standard Transformer with support for AdaLN-zero blocks"""

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        depth,
        heads,
        dim_head,
        mlp_dim,
        dropout=0.0,
        block_class=Block,
    ):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.layers = nn.ModuleList([])

        self.input_proj = (
            nn.Linear(input_dim, hidden_dim)
            if input_dim != hidden_dim
            else nn.Identity()
        )

        self.cond_proj = (
            nn.Linear(input_dim, hidden_dim)
            if input_dim != hidden_dim
            else nn.Identity()
        )

        self.output_proj = (
            nn.Linear(hidden_dim, output_dim)
            if hidden_dim != output_dim
            else nn.Identity()
        )

        for _ in range(depth):
            self.layers.append(
                block_class(hidden_dim, heads, dim_head, mlp_dim, dropout)
            )

    def forward(self, x, c=None, msg_keys=None, msg_mask=None, return_attention=False):

        if hasattr(self, "input_proj"):
            x = self.input_proj(x)
        if c is not None and hasattr(self, "cond_proj"):
            c = self.cond_proj(c)

        all_attn_weights = []

        for block in self.layers:
            if isinstance(block, MessageConditionedBlock):
                if return_attention:
                    x, attn = block(x, c, msg_keys=msg_keys, msg_mask=msg_mask, return_attention=True)
                    all_attn_weights.append(attn)
                else:
                    x = block(x, c, msg_keys=msg_keys, msg_mask=msg_mask)
            elif isinstance(block, ConditionalBlock):
                x = block(x, c)
            else:
                x = block(x)

        x = self.norm(x)
        if hasattr(self, "output_proj"):
            x = self.output_proj(x)

        if return_attention:
            return x, all_attn_weights
        return x


In [ ]:
#| export
class Embedder(nn.Module):
    def __init__(
        self,
        input_dim=10,
        smoothed_dim=10,
        emb_dim=10,
        mlp_scale=4,
    ):
        super().__init__()
        self.patch_embed = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1, stride=1)
        self.embed = nn.Sequential(
            nn.Linear(smoothed_dim, mlp_scale * emb_dim),
            nn.SiLU(),
            nn.Linear(mlp_scale * emb_dim, emb_dim),
        )

    def forward(self, x):
        """
        x: (B, T, D)
        """
        x = x.float()
        x = x.permute(0, 2, 1)
        x = self.patch_embed(x)
        x = x.permute(0, 2, 1)
        x = self.embed(x)
        return x



In [15]:
#| export
class MLP(nn.Module):
    """Simple MLP with optional normalization and activation"""

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim=None,
        norm_fn=nn.LayerNorm,
        act_fn=nn.GELU,
    ):
        super().__init__()
        norm_fn = norm_fn(hidden_dim) if norm_fn is not None else nn.Identity()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            norm_fn,
            act_fn(),
            nn.Linear(hidden_dim, output_dim or input_dim),
        )

    def forward(self, x):
        """
        x: (B*T, D)
        """
        return self.net(x)



In [ ]:
#| export
import math
class ARPredictor(nn.Module):
    def __init__(
        self,
        *,
        num_frames,
        depth,
        heads,
        mlp_dim,
        input_dim,
        hidden_dim,
        output_dim=None,
        dim_head=64,
        dropout=0.0,
        emb_dropout=0.0,
        msg_embedding_dim=256,   # VQ-VAE embedding_dim
        num_embeddings=256,      # VQ-VAE codebook size
        num_msg_tokens=49,       # tokens per message (the sender's 7 x 7 view)
        max_offset=14,           # relative poses: up to this many cells ahead/behind and right/left (the grid's size - 1);
                                 # a token's cell can be its view's depth farther
        max_age=8,               # messages kept this many steps after they were sent
    ):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, num_frames, input_dim))
        self.dropout = nn.Dropout(emb_dropout)
        self.num_msg_tokens, self.max_offset, self.max_age = num_msg_tokens, max_offset, max_age

        self.msg_token_embedding = nn.Embedding(num_embeddings, msg_embedding_dim)   # initialized from the VQ codebook
        self.msg_proj = (
            nn.Linear(msg_embedding_dim, hidden_dim)
            if msg_embedding_dim != hidden_dim
            else nn.Identity()
        )
        self.msg_pos_spatial = nn.Parameter(torch.randn(1, num_msg_tokens, hidden_dim) * 0.02)
        # each token's cell in the receiver's frame (`tile_positions`): cells ahead and to the right of the receiver
        view = math.isqrt(num_msg_tokens)                           # the sender's square view, the sender at its bottom centre
        self.reach = max_offset + view - 1
        row, col = torch.arange(num_msg_tokens) // view, torch.arange(num_msg_tokens) % view
        self.register_buffer("tile_cells", torch.stack([view - 1 - row, col - view // 2], -1), persistent=False)
        self.tile_ahead = nn.Embedding(2 * self.reach + 1, hidden_dim)
        self.tile_right = nn.Embedding(2 * self.reach + 1, hidden_dim)
        self.msg_heading = nn.Embedding(4, hidden_dim)               # the sender's heading and the age: added to all
        self.msg_age = nn.Embedding(max_age + 1, hidden_dim)         # of a message's tokens
        for e in (self.tile_ahead, self.tile_right, self.msg_heading, self.msg_age):
            nn.init.normal_(e.weight, std=0.02)
        self.no_msg = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)  # 'nothing received': always attendable

        self.transformer = Transformer(
            input_dim,
            hidden_dim,
            output_dim or input_dim,
            depth,
            heads,
            dim_head,
            mlp_dim,
            dropout,
            block_class=MessageConditionedBlock,
        )

    def tile_positions(self, pose):
        """Each token's cell relative to the receiver: (cells ahead, cells to the right) `(..., L, 2)`, from the sender's
        pose relative to the receiver `(..., 3)` (`relative_pose`): its cell in the sender's view, turned by the sender's
        heading, from the sender's cell."""
        a, r = self.tile_cells[:, 0], self.tile_cells[:, 1]          # in the sender's frame
        turn = (pose[..., 2] % 4).unsqueeze(-1)                       # clockwise quarter turns
        ahead = torch.stack([a, -r, -a, r])[turn.squeeze(-1)]        # (..., L)
        right = torch.stack([r, a, -r, -a])[turn.squeeze(-1)]
        return torch.stack([pose[..., 0:1] + ahead, pose[..., 1:2] + right], -1)

    def message_keys(self, messages):
        """Each step's message tokens and mask, as the blocks read them.
        messages: tokens (B, T, N, L), received (B, T, N), age (B, T, N), pose (B, T, N, 3) — `latest_messages`'
        returns: keys (B*T, N*L + 1, D) and mask (B*T, N*L + 1), True where nothing was received"""
        tokens, received = messages["tokens"], messages["received"]
        B, T, N, L = tokens.shape
        pose, age, m = messages["pose"], messages["age"], self.reach
        cells = self.tile_positions(pose).clamp(-m, m) + m           # (B, T, N, L, 2)
        about = self.msg_heading(pose[..., 2] % 4) + self.msg_age(age.clamp(0, self.max_age))   # (B, T, N, D)
        keys = (self.msg_proj(self.msg_token_embedding(tokens)) + self.msg_pos_spatial
                + self.tile_ahead(cells[..., 0]) + self.tile_right(cells[..., 1]) + about.unsqueeze(-2))
        keys = rearrange(keys, "b t n l d -> (b t) (n l) d")
        keys = torch.cat([keys, self.no_msg.expand(B * T, 1, -1)], dim=1)
        mask = rearrange(~received, "b t n -> (b t) n").repeat_interleave(L, dim=1)
        mask = torch.cat([mask, mask.new_zeros(B * T, 1)], dim=1)
        return keys, mask

    def forward(self, x, c, messages=None, return_attention=False):
        """
        x:        (B, T, D)  — state embeddings
        c:        (B, T, D)  — action embeddings
        messages: dict, optional — each step's latest received message per sender (`latest_messages`):
                  tokens (B, T, N, 49), received (B, T, N), age (B, T, N), pose (B, T, N, 3)
        """
        T = x.size(1)
        x = x + self.pos_embedding[:, :T]
        x = self.dropout(x)

        keys, mask = self.message_keys(messages) if messages is not None else (None, None)
        return self.transformer(x, c, msg_keys=keys, msg_mask=mask, return_attention=return_attention)

In [ ]:
# #| export
# class ARPredictor(nn.Module):
#     def __init__(
#         self,
#         *,
#         num_frames,
#         depth,
#         heads,
#         mlp_dim,
#         input_dim,
#         hidden_dim,
#         output_dim=None,
#         dim_head=64,
#         dropout=0.0,
#         emb_dropout=0.0,
#         msg_embedding_dim=256,   # VQ-VAE embedding_dim
#         num_embeddings=256,      # VQ-VAE codebook size
#     ):
#         super().__init__()
#         self.pos_embedding = nn.Parameter(torch.randn(1, num_frames, input_dim))
#         # self.pos_embedding_msg = nn.Parameter(torch.randn(1, num_frames, input_dim))
#         self.dropout = nn.Dropout(emb_dropout)

#         # Project message tokens to hidden_dim if needed
#         self.msg_proj = (
#             nn.Linear(msg_embedding_dim, hidden_dim)
#             if msg_embedding_dim != hidden_dim
#             else nn.Identity()
#         )

#         # Embed raw VQ indices → continuous vectors for cross-attention
#         self.msg_token_embedding = nn.Embedding(num_embeddings+1, msg_embedding_dim, padding_idx=num_embeddings) # can resue VQ-VAE's embedding layer

#         self.transformer = Transformer(
#             input_dim,
#             hidden_dim,
#             output_dim or input_dim,
#             depth,
#             heads,
#             dim_head,
#             mlp_dim,
#             dropout,
#             block_class=MessageConditionedBlock,  # ← use new block
#         )

#     def forward(self, x, c, msg_indices=None, return_attention=False):
#         """
#         x:           (B, T, D)    — state embeddings
#         c:           (B, T, A)    — action embeddings
#         msg_indices: (B, T, 49)      — VQ token indices from sender, optional
#         """
#         T = x.size(1)
#         x = x + self.pos_embedding[:, :T]
#         x = self.dropout(x)

#         # Embed message indices → token vectors
#         msg_tokens = None
#         if msg_indices is not None:
#             msg_indices = rearrange(msg_indices, "b t n -> (b t) n")  # flatten batch and time for embedding
#             msg_tokens = self.msg_token_embedding(msg_indices)  # (B, 49, msg_embedding_dim)
#             msg_tokens = self.msg_proj(msg_tokens)              # (B, 49, D)
#             msg_tokens = rearrange(msg_tokens, "(b t) n d -> b t n d", b=x.size(0), t=T)  # reshape back to (B, T, 49, D)

#         if return_attention:
#             x, attn_weights, gates = self.transformer(
#                 x, c, msg_tokens=msg_tokens, return_attention=True
#             )
#             return x, attn_weights, gates

#         x = self.transformer(x, c, msg_tokens=msg_tokens)
#         return x
    

In [ ]:
#| hide
torch.manual_seed(0)
B, T, N, D = 4, 3, 2, 64
model = ARPredictor(num_frames=T, depth=2, heads=4, mlp_dim=128, input_dim=D, hidden_dim=D,
                    msg_embedding_dim=32, max_age=4).eval()
x, c = torch.randn(B, T, D), torch.randn(B, T, D)
sent = torch.randint(0, 256, (B, 6, N, 49))                         # 6 steps of messages from 2 senders
arrived = torch.rand(B, 6, N) < 0.5
senders = torch.cat([torch.randint(0, 15, (B, 6, N, 2)), torch.randint(0, 4, (B, 6, N, 1))], -1)   # x, y, direction
receivers = torch.cat([torch.randint(0, 15, (B, 6, 2)), torch.randint(0, 4, (B, 6, 1))], -1)
msgs = latest_messages(sent, arrived, senders, receivers, steps=T, max_age=4)

In [ ]:
#| hide
# tile_positions: a cell the sender sees, placed from the receiver, is where it is: every heading, random poses
g = torch.Generator().manual_seed(1)
vec = torch.tensor(DIR_VECTORS)
for _ in range(200):
    rx, sx = torch.randint(0, 15, (2,), generator=g), torch.randint(0, 15, (2,), generator=g)
    rd, sd = torch.randint(0, 4, (), generator=g), torch.randint(0, 4, (), generator=g)
    i = int(torch.randint(0, 49, (), generator=g))
    ahead_s, right_s = 6 - i // 7, i % 7 - 3                          # token i: its cell in the sender's view
    s_ahead, s_right = vec[sd], torch.stack([-vec[sd][1], vec[sd][0]])
    cell = sx + ahead_s * s_ahead + right_s * s_right                  # where it is on the grid
    truth = relative_pose(cell, rd, rx, rd)[:2]                         # as the receiver sees it
    assert model.tile_positions(relative_pose(sx, sd, rx, rd))[i].tolist() == truth.tolist()

In [ ]:
#| hide
blind = model(x, c)
assert model(x, c, msgs).shape == (B, T, D)
assert torch.allclose(model(x, c, msgs), blind)                     # a new model starts as the blind one

for block in model.transformer.layers:                              # as if trained: the messages are read
    nn.init.normal_(block.cross_attn.out_proj.weight, std=0.1)
heard = model(x, c, msgs)
assert not torch.allclose(heard, blind)

In [ ]:
#| hide
# a message that did not arrive is not read
other = {**msgs, "tokens": torch.where(msgs["received"][..., None], msgs["tokens"], (msgs["tokens"] + 1) % 256)}
assert torch.allclose(model(x, c, other), heard)
# causal: the last step's messages change its prediction only
late = {k: v.clone() for k, v in msgs.items()}
late["tokens"][:, -1] = (late["tokens"][:, -1] + 7) % 256
late["received"][:, -1] = True
out = model(x, c, late)
assert torch.allclose(out[:, :-1], heard[:, :-1]) and not torch.allclose(out[:, -1], heard[:, -1])
# nothing received at all: only the 'nothing received' token is read
assert torch.isfinite(model(x, c, {**msgs, "received": torch.zeros_like(msgs["received"])})).all()

In [ ]:
#| hide
out, attn = model(x, c, msgs, return_attention=True)
assert torch.allclose(out, heard, atol=1e-5) and len(attn) == 2 and attn[0].shape == (B, T, N * 49 + 1)

In [ ]:
#| export
"""JEPA Implementation"""

def detach_clone(v):
    return v.detach().clone() if torch.is_tensor(v) else v

class JEPA(nn.Module):

    def __init__(
        self,
        encoder,
        predictor,
        action_encoder,
        projector=None,
        pred_proj=None,
        **kwargs
    ):
        super().__init__()

        self.encoder = encoder
        self.predictor = predictor
        self.action_encoder = action_encoder
        self.projector = projector or nn.Identity()
        self.pred_proj = pred_proj or nn.Identity()
        self.sigreg = SIGReg(knots=17, num_proj=1024)

    def encode(self, info):
        """Encode observations and actions into embeddings.
        info: dict with pixels and action keys
        """
        pixels = info['pixels']
        b = pixels.size(0)
        pixels = rearrange(pixels, "b t ... -> (b t) ...") # flatten for encoding
        output = self.encoder(pixels, interpolate_pos_encoding=True)
        pixels_emb = output.last_hidden_state[:, 0]  # cls token
        emb = self.projector(pixels_emb)
        info["emb"] = rearrange(emb, "(b t) d -> b t d", b=b)

        if "action" in info:
            info["act_emb"] = self.action_encoder(info["action"])#.to(device))

        return info

    def predict(self, emb, act_emb, messages=None):
        """Predict next state embedding
        emb: (B, T, D) 
        act_emb: (B, T, A_emb)
        messages: each step's received messages (`ARPredictor`), optional
        """
        preds = self.predictor(emb, act_emb, messages=messages)
        preds = self.pred_proj(rearrange(preds, "b t d -> (b t) d"))
        preds = rearrange(preds, "(b t) d -> b t d", b=emb.size(0))
        return preds
    

    def loss_fn(self, output, pred_emb, target_emb, emb, lambd):
        output["pred_loss"] = (pred_emb - target_emb).pow(2).mean()
        output["sigreg_loss"]= self.sigreg(emb.transpose(0, 1))
        output["jepa_loss"] = output["pred_loss"] + lambd * output["sigreg_loss"] 
        return output

    ####################
    ## Inference only ##
    ####################

    
    def rollout(self, info, action_sequence, history_size: int = 3):
        """Rollout the model given an initial info dict and action sequence.
        pixels: (B, S, H, C, H, W) the observed steps
        action_sequence: (B, S, T) the actions, the observed steps' first
        messages: dict, optional — the observed steps' received messages (`latest_messages`), each (B, S, H, ...).
            The imagined steps keep the last observed step's messages, older by a step each (none arrive in them;
            a message older than the predictor's max_age is dropped), placed from the last observed pose (the
            imagined poses are unknown)
        - S is the number of action plan samples
        - T is the time horizon
        """

        assert "pixels" in info, "pixels not in info_dict"
        H = info["pixels"].size(2)
        B, S, T = action_sequence.shape[:3]
        act_0, act_future = torch.split(action_sequence, [H, T - H], dim=2)
        info["action"] = act_0
        info["observed"] = H
        n_steps = T - H

        _init = {k: v[:, 0] for k, v in info.items() if torch.is_tensor(v)}
        _init = self.encode(_init)
        emb = info["emb"] = _init["emb"].unsqueeze(1).expand(B, S, -1, -1)
        _init = {k: detach_clone(v) for k, v in _init.items()}

        emb = rearrange(emb, "b s ... -> (b s) ...").clone()
        act = rearrange(act_0, "b s ... -> (b s) ...")
        act_future = rearrange(act_future, "b s ... -> (b s) ...")

        messages = info.get("messages")
        if messages is not None:
            messages = {k: rearrange(v, "b s ... -> (b s) ...") for k, v in messages.items()}
        max_age = self.predictor.max_age

        def context(n):
            "The messages of the last HS of the first n steps (the steps after the observed ones: imagined)."
            if messages is None:
                return None
            steps = []
            for k in range(max(0, n - HS), n):
                snap = {key: v[:, min(k, H - 1)] for key, v in messages.items()}
                if k >= H:
                    age = snap["age"] + (k - H + 1)
                    snap["received"] = snap["received"] & (age <= max_age)
                    snap["age"] = age.clamp(max=max_age)
                steps.append(snap)
            return {key: torch.stack([s[key] for s in steps], dim=1) for key in steps[0]}

        HS = history_size
        for t in range(n_steps + 1):
            act_emb = self.action_encoder(act)
            emb_trunc = emb[:, -HS:]
            act_trunc = act_emb[:, -HS:]
            pred_emb = self.predict(emb_trunc, act_trunc, context(emb.size(1)))[:, -1:]
            emb = torch.cat([emb, pred_emb], dim=1)

            if t < n_steps:
                next_act = act_future[:, t : t + 1]
                act = torch.cat([act, next_act], dim=1)

        pred_rollout = rearrange(emb, "(b s) ... -> b s ...", b=B, s=S)
        info["predicted_emb"] = pred_rollout

        return info


    # def criterion(self, info_dict: dict):
    #     """Compute the cost between predicted embeddings and goal embeddings."""
    #     pred_emb = info_dict["predicted_emb"]  # (B,S, T-1, dim)
    #     goal_emb = info_dict["goal_emb"]  # (B, S, T, dim)

    #     goal_emb = goal_emb[..., -1:, :].expand_as(pred_emb)

    #     # return last-step cost per action candidate
    #     cost = F.mse_loss(
    #         pred_emb[..., -1:, :],
    #         goal_emb[..., -1:, :].detach(),
    #         reduction="none",
    #     ).sum(dim=tuple(range(2, pred_emb.ndim)))  # (B, S)

    #     return cost

    def criterion(self, info_dict: dict):
        """Cost = min over imagined steps of embedding distance to goal."""
        pred_emb = info_dict["predicted_emb"]      # (B, S, T, D)
        goal_emb = info_dict["goal_emb"]           # (B, S=1, T=1, D)

        goal = goal_emb[..., -1:, :].expand_as(pred_emb).detach()
        per_step = F.mse_loss(pred_emb, goal, reduction="none").sum(-1)  # (B, S, T)

        # exclude the history steps: only score imagined future states
        H = info_dict.get("observed", 3)  # the observed steps (rollout's)
        per_step = per_step[..., H:]

        cost = per_step.min(dim=-1).values          # (B, S)
        return cost

    def get_cost(self, info_dict: dict, action_candidates: torch.Tensor):
        """ Compute the cost of action candidates given an info dict with goal and initial state."""

        assert "goal" in info_dict, "goal not in info_dict"

        device = next(self.parameters()).device
        for k in list(info_dict.keys()):
            if torch.is_tensor(info_dict[k]):
                info_dict[k] = info_dict[k].to(device)

        goal = {k: v[:, :1] for k, v in info_dict.items() if torch.is_tensor(v)}
        goal["pixels"] = goal["goal"]

        for k in info_dict:
            if k.startswith("goal_"):
                goal[k[len("goal_") :]] = goal.pop(k)

        goal.pop("action")
        goal = self.encode(goal)
        info_dict["goal_emb"] = goal["emb"].unsqueeze(2)  # (B, S=1, T=1, D) — ready to broadcast in criterion
        info_dict = self.rollout(info_dict, action_candidates)

        cost = self.criterion(info_dict)
        
        return cost
    

In [23]:
#| hide
from stable_pretraining.backbone.utils import vit_hf

encoder = vit_hf(
    size="tiny",
    patch_size=14,
    image_size=224,
    pretrained=False,
    use_mask_token=True,
)

21:16:38 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}


In [47]:
#| hide
num_frames = 3
depth = 6
heads = 8
mlp_dim = 2048
input_dim = 192
hidden_dim = 192

predictor = ARPredictor(
    num_frames=num_frames,
    depth=depth,
    heads=heads,
    mlp_dim=mlp_dim,
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    output_dim=input_dim
)


In [40]:
#| hide
act_enc = DiscreteActionEncoder(num_actions=4, action_emb_dim=192)

In [41]:
#| hide
projector = MLP(input_dim=192, hidden_dim=2048, output_dim=192, norm_fn=torch.nn.BatchNorm1d)
pred_proj = MLP(input_dim=192, hidden_dim=2048, output_dim=192, norm_fn=torch.nn.BatchNorm1d)

In [48]:
#| hide
world_model = JEPA(
    encoder=encoder,
    predictor=predictor,
    action_encoder=act_enc,
    projector=projector,
    pred_proj=pred_proj,
)

In [49]:
#| hide
batch = 3
batch_obs = torch.randn(batch, num_frames, 3, 224, 224)
batch_actions = torch.randint(0, 4, (batch, num_frames, 1))  # assuming action_emb_dim = input_dim for this test

output = world_model.encode({"pixels": batch_obs, "action": batch_actions})

In [50]:
#| hide
[output[k].shape for k in output if torch.is_tensor(output[k])]

[torch.Size([3, 3, 3, 224, 224]),
 torch.Size([3, 3, 1]),
 torch.Size([3, 3, 192]),
 torch.Size([3, 3, 1, 192])]

In [ ]:
#| hide
ctx_len = num_frames
n_preds = 1
sent = torch.randint(0, 256, (batch, num_frames, 1, 49))            # one sender
msgs = latest_messages(sent, torch.ones(batch, num_frames, 1, dtype=torch.bool), torch.zeros(batch, num_frames, 1, 3, dtype=torch.long),
                       torch.zeros(batch, num_frames, 3, dtype=torch.long), steps=ctx_len, max_age=8)

emb = output["emb"]  # (B, T+1, D)
act_emb = output["act_emb"].squeeze(2)  # (B, T, D_act)

ctx_emb = emb[:, :ctx_len]
ctx_act = act_emb[:, : ctx_len]

tgt_emb = emb[:, n_preds:] # label
pred_emb = world_model.predict(ctx_emb, ctx_act, msgs) # pred 

In [53]:
#| hide
pred_emb.shape

torch.Size([3, 3, 192])

In [ ]:
#| hide
# planning: the imagined steps keep the last observed step's messages (older a step at each), so they still count
world_model.eval()
for block in world_model.predictor.transformer.layers:
    nn.init.normal_(block.cross_attn.out_proj.weight, std=0.1)
S, H, horizon = 2, 2, 5
pixels = torch.randn(1, S, H, 3, 224, 224)
plans = torch.randint(0, 4, (1, S, horizon))
messages = {"tokens": torch.randint(0, 256, (1, S, H, 1, 49)), "received": torch.ones(1, S, H, 1, dtype=torch.bool),
            "age": torch.zeros(1, S, H, 1, dtype=torch.long), "pose": torch.zeros(1, S, H, 1, 3, dtype=torch.long)}
with torch.no_grad():
    heard = world_model.rollout({"pixels": pixels, "messages": messages}, plans)
    blind = world_model.rollout({"pixels": pixels}, plans)
assert heard["predicted_emb"].shape == (1, S, horizon + 1, 192) and heard["observed"] == H
assert torch.allclose(heard["predicted_emb"][:, :, :H], blind["predicted_emb"][:, :, :H])     # the observed steps
assert not torch.allclose(heard["predicted_emb"][:, :, -1], blind["predicted_emb"][:, :, -1])  # the last imagined one

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()